# Buisness Problem

## A Retail store is required to analyze the day-to-day transactions and keep a track of its customers spread across various locations along with their purchases/returns across various categories.
## Create a report and display the below calculated metrics, reports and inferences.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import seaborn as sns

In [ ]:
cust = pd.read_csv('./Customer.csv')
products = pd.read_csv('./prod_cat_info.csv')
tran = pd.read_csv('./Transactions.csv')

In [ ]:
cust

In [ ]:
products

In [ ]:
tran

## Joining Dataset

In [ ]:
cust_tran = pd.merge(cust,tran, left_on = cust.customer_Id , right_on = tran.cust_id)

In [ ]:
cust_tran.drop(columns='key_0',inplace=True)

In [ ]:
Customer_Final = pd.merge(cust_tran , products , left_on = [cust_tran.prod_subcat_code , cust_tran.prod_cat_code] , right_on = [products.prod_sub_cat_code , products.prod_cat_code])

In [ ]:
Customer_Final.drop(columns='cust_id',inplace=True)

## EDA & Cleaning

In [ ]:
Customer_Final.info()

In [ ]:
Customer_Final.head(10)

In [ ]:
Customer_Final.tail(10)

In [ ]:
c = ['min', 'Q1', 'median', 'Q3' , 'max']
five_number_summary = Customer_Final.describe().astype(int).iloc[3:,3:7].T
five_number_summary.columns = c
five_number_summary

In [ ]:
Customer_Final.drop(columns = ['key_0',	'key_1'],inplace = True)

In [ ]:
Customer_Final.rename({'prod_cat_code_y':'prod_cat_code'},inplace=True)

In [ ]:
col = Customer_Final.select_dtypes('O').drop(columns=['DOB','tran_date']).columns

In [ ]:
for i in col:
    print(Customer_Final[i].value_counts())
    print('\n')

In [ ]:
Customer_Final.drop(columns=['prod_subcat_code' , 'prod_cat_code_x'],inplace = True)

In [ ]:
Customer_Final[['customer_Id','city_code','transaction_id','prod_cat_code_y','prod_sub_cat_code']]=Customer_Final[['customer_Id','city_code','transaction_id','prod_cat_code_y','prod_sub_cat_code']].astype('object')

In [ ]:
Customer_Final.DOB = pd.to_datetime(Customer_Final.DOB,format = '%d-%m-%Y')

In [ ]:
Customer_Final.tran_date = pd.to_datetime(Customer_Final.tran_date,format = 'mixed')

In [ ]:
Customer_Final.select_dtypes(include='number').hist()
plt.show()

In [ ]:
fig , axs = plt.subplots(2,2)
Customer_Final.Gender.value_counts().plot(kind = 'bar',ax=axs[0,0])
Customer_Final.Store_type.value_counts().plot(kind = 'bar',ax=axs[0,1])
Customer_Final.prod_cat.value_counts().plot(kind = 'bar',ax=axs[1,0])
Customer_Final.prod_subcat.value_counts().plot(kind = 'bar',ax=axs[1,1])
plt.show()

In [ ]:
Customer_Final.info()

In [ ]:
Customer_Final.describe().T

In [ ]:
time_period = Customer_Final.tran_date.max() - Customer_Final.tran_date.min()

### Total Time Period

In [ ]:
time_period

### No. Of Returns

In [ ]:
Customer_Final.loc[Customer_Final['total_amt']  < 0 ,'total_amt'].count()

### Popular Product Categories in Male and Female

In [ ]:
pd.DataFrame(Customer_Final.groupby(['prod_cat','Gender']).agg({'total_amt':'sum','Qty':'sum'}).sort_values(by=['total_amt','Qty'],ascending=False)).iloc[0:2,:]

### City Code With Highest Customers

In [ ]:
pd.DataFrame(Customer_Final.groupby('city_code').customer_Id.count()/Customer_Final.shape[0]).sort_values(by='customer_Id',ascending=False).iloc[0:1,:]

### Most Selling Store Type

In [ ]:
Customer_Final.groupby('Store_type').agg({'total_amt':'sum','customer_Id':'count'}).astype('int').sort_values(by=['total_amt','customer_Id'],ascending=False).iloc[0:1,:]

### Total amount earned from the "Electronics" and "Clothing" categories from Flagship Stores?

In [ ]:
Customer_Final.loc[(Customer_Final['Store_type'] == 'Flagship store')&(Customer_Final['prod_cat'].isin(['Electronics','Clothing']))].agg({'total_amt':'sum'})

### Total amount earned from "Male" customers under the "Electronics" category?

In [ ]:
Customer_Final.loc[(Customer_Final.Gender == 'M') & (Customer_Final.prod_cat == 'Electronics')].agg({'total_amt' : 'sum'})

### How many customers have more than 10 unique transactions, after removing all transactions which have any negative amounts?

In [ ]:
## customers have more than 10 unique transactions
cust = pd.DataFrame(Customer_Final.customer_Id.value_counts() > 10)

In [ ]:
## Customers with positive Transactions
p_cust = Customer_Final.loc[Customer_Final['total_amt'] > 0]

In [ ]:
## Filtering the CustomerID
customers_with_10_plus = cust.loc[cust['count'] == 1]

In [ ]:
## Total Amount Of the customers which have positive Transactions and made more than 10 Transactions
p_cust[p_cust['customer_Id'].isin(customers_with_10_plus.index)].total_amt.sum()

## For all customers aged between 25 - 35, find out:

### a. What was the total amount spent for “Electronics” and “Books” product categories?
### b. What was the total amount spent by these customers between 1st Jan, 2014 to 1st Mar, 2014?


In [ ]:
## Getting the Age From DOB Column
Customer_Final['Age'] = Customer_Final.tran_date.max() - Customer_Final['DOB']

In [ ]:
Customer_Final['Age'] = Customer_Final.Age.dt.days

In [ ]:
Customer_Final.Age = Customer_Final.Age / 365.25

In [ ]:
Customer_Final.Age = Customer_Final.Age.astype('int')

In [ ]:
middle_age_cust = Customer_Final.loc[(Customer_Final.Age >= 25) & (Customer_Final.Age <= 35)]

In [ ]:
## total amount spent for “Electronics” and “Books” product categories?
middle_age_cust.loc[middle_age_cust['prod_cat'].isin(['Electronics','Books'])].total_amt.sum()

In [ ]:
## total amount spent by these customers between 1st Jan, 2014 to 1st Mar, 2014?
middle_age_cust.loc[(middle_age_cust['tran_date'] >= '2014-01-01') & (middle_age_cust['tran_date'] <= '2014-03-01')].total_amt.sum()